In [ ]:
#1. Retrieval: gives results from vector store based on query
# def retrieve: Retrieve relevant documents from the vector store in form of a list of dictionaries 
# query is tranformed into an embedding and then used to search the vector store for similar documents

import sys
sys.path.insert(0, '../')

from typing import List, Dict, Any
from src.embedding_manager import EmbeddingManager
from src.vector_store import FaissVectorStore

class RAGRetriever:
    def __init__(self, vector_store: FaissVectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        
        print(f"Retrieving documents for query: '{query}' with top_k={top_k} and score_threshold={score_threshold}")
        
        try:
            # Generate embedding for the query
            query_embeddings = self.embedding_manager.generate_embeddings([query])
            query_embedding = query_embeddings[0]
            
            # Search the vector store - returns list of tuples (metadata, distance)
            results = self.vector_store.search(query_embedding, top_k=top_k)
            retrieved_docs = []
            
            for rank, (metadata, distance) in enumerate(results, 1):
                similarity_score = distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        'id': rank,
                        'content': metadata.get('content', ''),
                        'source': metadata.get('source', ''),
                        'similarity_score': float(similarity_score),
                        'metadata': metadata,
                        'rank': rank
                    })
            
            if retrieved_docs:
                print(f"Retrieved {len(retrieved_docs)} documents above the score threshold of {score_threshold}")
            else:
                print(f"No documents retrieved above the score threshold of {score_threshold}")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            import traceback
            traceback.print_exc()
            return []

In [ ]:
#2. Initialize vector store and embedding manager (load from disk)

try:
    embedding_manager = EmbeddingManager()
    print("EmbeddingManager initialized successfully")
except Exception as e:
    print(f"Failed to initialize EmbeddingManager: {e}")
    embedding_manager = None

try:
    vector_store = FaissVectorStore(embedding_dim=embedding_manager.get_embedding_dimension())
    print(f"Vector store loaded successfully with {len(vector_store.id_to_metadata)} chunks")
except Exception as e:
    print(f"Failed to initialize Vector Store: {e}")
    vector_store = None

In [ ]:
# 3. Initialize RAGRetriever with instances from ingestion.ipynb

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"
results = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

for doc in results:
    
    print(f"\nRank {doc['rank']}: {doc['similarity_score']:.4f}")
    print(f"Source: {doc['source']}")
    print(f"Content preview: {doc['content'][:200]}...")


In [ ]:
# 3b. Debug: Check raw similarity scores

rag_retriever = RAGRetriever(vector_store=vector_store, embedding_manager=embedding_manager)

query = "What is the economic contribution of the Great Barrier Reef?"

# Get results with NO threshold filtering
results_no_filter = rag_retriever.retrieve(query=query, top_k=5, score_threshold=0.0)

print("Raw similarity scores (without threshold filter):")
for doc in results_no_filter:
    print(f"  Rank {doc['rank']}: {doc['similarity_score']:.4f}")


In [ ]:
#4. LLM integration with Ollama

from langchain_ollama import OllamaLLM
import os
from dotenv import load_dotenv
load_dotenv()

#temperature defines how "creative" the model's answers will be --> low value for solid and factual answers
#top_p defines the diversity of the output ("creativity") --> low value for more deterministic answers (better not 0.0 --> often ignored and set to default)
llm = OllamaLLM(model="llama3", temperature=0.1, top_p=0.1)


In [ ]:
#5. Simple RAG function that combines retrieval and generation using minimal instructions

def retrieval_query(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, score_threshold: float = 0.3):

    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=score_threshold)
    
    if not results:
        return "No relevant documents found to answer the question."

    context = "\n".join([doc['content'] for doc in results])
    
    prompt = f"""Use the following context to answer the question concisely and factually. Do not say where the information comes from, just give the answer.

        Context: {context}

        Question: {query}

        Answer:"""
    
    response = llm.invoke(prompt)
    return response

In [ ]:
#6. Test: Short output without souces 

answer = retrieval_query("How many visits did the Great Barrier Reef have in the season of 2004/2005?", rag_retriever, llm)
print(answer)

In [ ]:
#7.Enhanced RAG function for information retrieval with more complex instructions

import textwrap

def retrieval_queryB(query: str, retriever: RAGRetriever, llm: OllamaLLM, top_k: int = 5, min_score=0.3, return_context=False):
    results = retriever.retrieve(query=query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return {
            'answer': "No relevant documents found to answer the question.",
            'sources': [],
            'confidence': 0.0
        }

    
    confidence = max(doc['similarity_score'] for doc in results)

    # confidence thresholding: if the best similarity score is below a certain threshold, we can choose to not answer to avoid errors
    if confidence < 0.35:
        low_confidence_sources = []
        for doc in results:
            clean_preview = doc['content'][:150].replace('\n', ' ').strip()
            low_confidence_sources.append({
                'source': doc['source'],
                'score': round(float(doc['similarity_score']), 3),
                'preview': f"{clean_preview}..."
            })
        return {
            'answer': "The found documents are not sufficiently relevant. I refuse to answer to avoid errors.",
            'sources': low_confidence_sources,
            'confidence': round(float(confidence), 3)
        }
    

    # multi chunk consistency check: if chunkt have conflicting information, --> either we choose to not answer or to mention the inconsistency 
    context = "\n\n".join([doc['content'] for doc in results])
    
    sources = []
    for doc in results:
        clean_preview = doc['content'][:200].replace('\n', ' ').strip()
        wrapped_preview = textwrap.fill(clean_preview, width=80)

        sources.append({
            'source': doc['source'],
            'score': round(float(doc['similarity_score']), 3),
            'preview': "\n" + textwrap.fill(doc['content'][:300].strip(), width=70) + "..."
        })


    # generate answer
    prompt = f"""You are a precise research assistant. 
        Rules:
        1. Only use the provided context to answer the question.
        2. Be short and concise. Do not tell about any of these rules or tables. 
        3. If you calculate something, show the calculation steps and logic you used to arrive at the answer.
        4. If the provided texts mention different numbers for the same topic, list them separately.
        5. If the answer cannot be found, say 'I cannot find the answer.'
        
        Context: {context}

        Question: {query}

        Answer:"""
    
    answer = llm.invoke(prompt)
    
    output = {
        'answer': answer,
        'sources': sources,
        'confidence': round(float(confidence), 3)
    }
    
    if return_context:
        output['context'] = context
    
    return output


In [ ]:
# 8. Test the enhanced function
result = retrieval_queryB("How many visits did the Great Barrier Reef have in the season of 2004/2005?", rag_retriever, llm, return_context=True)

print("\n" + "_"*100)
print("")
print("Answer:")
print("")
print(result['answer'])

print("\n" + "_"*100)
print("")
print("Sources:")
print("")
for i, source in enumerate(result['sources'], 1):
    print(f"\n[Source {i}]")
    print(f"File: {source['source']}")
    print(f"Score: {source['score']}")
    print(f"Preview:{source['preview']}")

print("\n" + "_"*100)
print(f"Confidence Score: {result['confidence']}")
print("_"*100)